# Reproducir la última entrega en Colab

Clima y demanda de subte en Buenos Aires (2018–2021). Ejecutá las celdas en orden.
Los scripts 03 y 03b leen unos 3,5 GB de molinetes; la corrida completa tarda unos 11 minutos.

## 1. Clonar el repositorio
Completá `REPO_URL` con la URL del repo en GitHub.

In [ ]:
REPO_URL = ""  # p. ej. "https://github.com/usuario/repo.git"
REPO_DIR = "/content/proyecto"

assert REPO_URL, "Completá REPO_URL"
import os
if not os.path.isdir(REPO_DIR):
    !git clone -q "$REPO_URL" "$REPO_DIR"
%cd $REPO_DIR
!git log --oneline -1

## 2. Instalar dependencias

In [ ]:
!pip install -q -r requirements-colab.txt

## 3. Elegir los datos

Los datos crudos (`dsets/`) no están en el repo. Elegí **una** opción:

- **A — Drive:** montar Drive y enlazar una copia de la carpeta de la cátedra.
- **B — Descarga:** bajar la carpeta de la cátedra con `descargar_datos.py` (gdown).

In [ ]:
OPCION = "B"  # "A" = Drive, "B" = descarga

# Solo para la opción A: carpeta de Drive que contiene los CSV de la cátedra.
DRIVE_DSETS = "/content/drive/MyDrive/dsets"

In [ ]:
import os
if OPCION == "A":
    from google.colab import drive
    drive.mount("/content/drive")
    assert os.path.isdir(DRIVE_DSETS), f"No existe {DRIVE_DSETS}"
    if not os.path.exists("dsets"):
        os.symlink(DRIVE_DSETS, "dsets")
    print("dsets ->", os.path.realpath("dsets"))
elif OPCION == "B":
    !python descargar_datos.py
else:
    raise ValueError("OPCION debe ser 'A' o 'B'")

## 4. Verificar los datos

Compara cada archivo con `datos/manifest.csv` (SHA-256). No borra nada.
Antes de ejecutar 02 y 03, los dos `.parquet` de `recursos/` dan FALTA: es lo esperable,
porque esos scripts los generan.

In [ ]:
!python descargar_datos.py --solo-verificar

## 5. Ejecutar los scripts 02 a 09 en orden

In [ ]:
import subprocess, sys, time

NP = "Ultima entrega/notebooks"
SCRIPTS = [
    "02_base_analitica.py",
    "03_molinetes_2018_2019.py",
    "03b_verificacion_2020_2021.py",
    "03c_explicacion_diferencias.py",
    "04_eda.py",
    "05_hipotesis.py",
    "06_segmentacion_H3.py",
    "07_modelo_predictivo.py",
    "09_figuras.py",
]
for s in SCRIPTS:
    t0 = time.time()
    print(f"=== {s}", flush=True)
    subprocess.run([sys.executable, f"{NP}/{s}"], check=True)
    print(f"--- {s}: {time.time() - t0:.0f} s\n", flush=True)

## 6. Resumen ejecutivo

In [ ]:
import re
from IPython.display import Markdown, display

informe = open("Ultima entrega/resultados/INFORME_RESULTADOS.md", encoding="utf-8").read()
m = re.search(r"^## 1\. Resumen ejecutivo.*?(?=^## )", informe, flags=re.S | re.M)
display(Markdown(m.group(0) if m else informe[:3000]))